# Pipeline Step 1: Data Preprocessing & ETL

## Inputs
- **SofaScore Player Match Stats**:
  - `IsraeliPremierLeague_23-24.xlsx`
  - `IsraeliPremierLeague_24-25.xlsx`
- **Israeli Premier League (ISA) Shot Events**:
  - `all_shots_check_fixed_raw.xlsx`
- **Player/Team Name Mapping Dictionary**:
  - `ligat_haal_player_match_cleaned.xlsx`

## Outputs
- **Preprocessed SofaScore Dataset**: `sofa_preprocessed.csv` (contains cleaned, normalized match-by-match statistics for all players)
- **Preprocessed Shot Events Dataset**: `shots_preprocessed.csv` (contains cleaned shot event records with player names mapped and disclosing attributes removed)


In [1]:
from __future__ import annotations

import os
import unicodedata
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")


Libraries loaded successfully.


### String Normalization Helper
Applies a standardized text cleanup for names, teams, and text categories, ensuring robust joins.

In [2]:
from typing import Union

def normalize_txt(s: Union[pd.Series, str]) -> Union[pd.Series, str]:
    """
    Applies string normalization: converts to lowercase, strips outer whitespace,
    replaces multiple whitespace with a single space, and removes diacritics.
    """
    if isinstance(s, pd.Series):
        return (s.astype(str)
                 .str.normalize("NFKD")
                 .str.encode("ascii", errors="ignore").str.decode("ascii")
                 .str.lower().str.strip()
                 .str.replace(r"\s+", " ", regex=True))
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return " ".join(s.lower().split())


### Preprocessing SofaScore Player Statistics
Normalizes player stats by minutes played, extracts count values from parenthesized strings, and cleans score outcomes.

In [3]:
def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies general preprocessing steps to the SofaScore player performance dataset.
    """
    # Convert positions to numerical scales
    POSITION_VOCAB = {'G': 0, 'D': 1/3, 'M': 2/3, 'F': 1}
    df = df.copy()

    df['Position'] = df['Position'].apply(lambda x: POSITION_VOCAB.get(x, None))

    # Drop irrelevant columns to prevent feature redundancy
    columns_to_drop = ['Duels (won)', 'Sofascore Rating', 'Notes Attacking', 'Defensive actions',
                       'Notes Defending', 'Notes Passing', 'Notes Goalkeeping']
    df = df.drop(columns=columns_to_drop, errors="ignore")

    # Convert minutes played to float
    df['Minutes played'] = (
        df['Minutes played'].astype(str)
        .str.replace(r"(\d+)'", r'\1', regex=True)
        .astype(float)
    )

    # Split parenthesized columns (e.g., 'Dribble attempts (successful)') into Total/Successful
    parenthesis_columns = [col for col in df.columns if '(' in col and ')' in col]
    if 'Expected Goals (xG)' in parenthesis_columns:
        parenthesis_columns.remove('Expected Goals (xG)')

    for column in parenthesis_columns:
        base_name = column.split(' (')[0].strip()
        total_col = f"{base_name} Total"
        succ_col  = f"{base_name} Successful"
        ext = df[column].astype(str).str.extract(r'(\d+)\s*\((\d+)\)')
        df[total_col] = pd.to_numeric(ext[0], errors='coerce')
        df[succ_col]  = pd.to_numeric(ext[1], errors='coerce')
        df.drop(columns=[column], inplace=True)

    # Split Accurate passes (e.g., '15/20') into attempts and successful passes
    if 'Accurate passes' in df.columns:
        ap = df['Accurate passes'].astype(str).str.extract(r'(\d+)\s*/\s*(\d+)')
        df['Successful Passes'] = pd.to_numeric(ap[0], errors='coerce')
        df['Pass Attempts']     = pd.to_numeric(ap[1], errors='coerce')
        df.drop(columns=['Accurate passes'], inplace=True)

    # Fill empty/missing metrics with zero
    df = df.fillna(0)

    # Normalize numerical feature counts by minutes played (per minute scale)
    non_normed = ['Player', 'Player_ID', 'Team', 'Position', 'Home/Away', 'Date', 'Score', 'Minutes played']
    feat_cols = [c for c in df.columns if c not in non_normed]
    df[feat_cols] = df[feat_cols].astype(float).div(df['Minutes played'], axis=0)

    # Scale minutes played to a fraction of a full 90-minute match
    df['Minutes played'] = df['Minutes played'] / 90.0

    # Extract goals scored for and against from match score
    def extract_goals(row):
        is_home = row['Home/Away']
        score_str = str(row['Score'])
        if '-' in score_str:
            gf, ga = score_str.split('-') if is_home else score_str.split('-')[::-1]
        else:
            gf, ga = 0, 0
        return pd.Series([int(gf), int(ga)])

    df[['Goals for', 'Goals against']] = df.apply(extract_goals, axis=1)

    # Convert Date into datetime
    df['Date'] = df['Date'].astype(str).str.zfill(8)
    df['Date'] = pd.to_datetime(df['Date'], format="%d%m%Y")

    return df


### Loading and Running SofaScore Processing
Load raw SofaScore season data, run preprocessing, merge the datasets, and save the result.

In [4]:
print("Loading SofaScore raw datasets...")
ipl23_24 = pd.read_excel("IsraeliPremierLeague_23-24.xlsx")
ipl24_25 = pd.read_excel("IsraeliPremierLeague_24-25.xlsx")

print("Preprocessing SofaScore raw datasets...")
processed_23_24 = preprocess_df(ipl23_24)
processed_24_25 = preprocess_df(ipl24_25)

# Concatenate seasons
sofa_preprocessed = pd.concat([processed_23_24, processed_24_25], ignore_index=True)
print(f"SofaScore preprocessed dataset shape: {sofa_preprocessed.shape}")

# Save to CSV
sofa_preprocessed.to_csv("sofa_preprocessed.csv", index=False)
print("Saved SofaScore preprocessed data to 'sofa_preprocessed.csv'.")


Loading SofaScore raw datasets...
Preprocessing SofaScore raw datasets...
SofaScore preprocessed dataset shape: (14818, 43)
Saved SofaScore preprocessed data to 'sofa_preprocessed.csv'.


### Name Mapping Logic
Defines the lookup logic to map shooter and goalie names in the ISA shot events database to their matching names in SofaScore, addressing differences in spellings and initials.

In [5]:
# Constants for mapping columns
COL_SHOT_SHOOTER_NAME = "shooter"
COL_SHOT_KEEPER_NAME  = "goalie"
COL_SHOT_TEAM         = "Team"

MAP_COL_TEAM_SHOT     = "Team_name_shot"
MAP_COL_PLAYER_SHOT   = "Player_name_shot"
MAP_COL_PLAYER_PERF   = "player_name_performance"
MAP_COL_TEAM_PERF     = "Team_name_performance"

def map_shots_names_to_sofa(
    shots_df: pd.DataFrame,
    mapping_xlsx_path: str
) -> pd.DataFrame:
    """
    Maps goalie names by name-only matching and shooter names by name + team matching.
    Falls back to goalie manual additions if mapping fails.
    """
    df = shots_df.copy()

    # Load mapping dictionary
    m = pd.read_excel(mapping_xlsx_path)
    req = {MAP_COL_TEAM_SHOT, MAP_COL_PLAYER_SHOT, MAP_COL_PLAYER_PERF, MAP_COL_TEAM_PERF}
    miss = req - set(m.columns)
    if miss:
        raise ValueError(f"Mapping file is missing required columns: {miss}")

    # Normalize all string columns for robust string matching
    for c in [COL_SHOT_SHOOTER_NAME, COL_SHOT_KEEPER_NAME, COL_SHOT_TEAM]:
        if c in df.columns:
            df[c] = normalize_txt(df[c])
    for c in req:
        m[c] = normalize_txt(m[c])

    # Store originals for mapping/fallback checks
    original_goalie = df[COL_SHOT_KEEPER_NAME].copy()
    original_shooter = df[COL_SHOT_SHOOTER_NAME].copy()

    # --- Goalkeeper Mapping (NAME ONLY) ---
    keeper_map = m[[MAP_COL_PLAYER_SHOT, MAP_COL_PLAYER_PERF]].drop_duplicates()
    df = df.merge(
        keeper_map,
        how="left",
        left_on=COL_SHOT_KEEPER_NAME,
        right_on=MAP_COL_PLAYER_SHOT,
        suffixes=("", "_keepmap"),
    )

    # Goalie Manual Additions dictionary
    manual_additions = {
        'K. Niemczycki': 'Karol Niemczycki',
        'M. Alchevski' : 'Marko Alchevski',
        'M. Zalmanovich': 'Matan Zalmanovich',
        'O. Antman'    : 'Ofek Antman',
        'S. Sluga'     : 'Simon Sluga',
        'T. Sultani'   : 'Tomas Sultani'
    }
    manual_additions_norm = {normalize_txt(k): normalize_txt(v) for k, v in manual_additions.items()}

    # Apply goalie mapping from file, fallback to manual additions, fallback to normalized original name
    df[COL_SHOT_KEEPER_NAME] = df[MAP_COL_PLAYER_PERF].fillna(original_goalie.map(manual_additions_norm)).fillna(original_goalie)
    df.drop(columns=[MAP_COL_PLAYER_SHOT, MAP_COL_PLAYER_PERF], inplace=True, errors='ignore')

    # --- Shooter Mapping (NAME + TEAM) ---
    shooter_map = m[[MAP_COL_PLAYER_SHOT, MAP_COL_TEAM_SHOT, MAP_COL_PLAYER_PERF]].drop_duplicates()
    df = df.merge(
        shooter_map,
        how="left",
        left_on=[COL_SHOT_SHOOTER_NAME, COL_SHOT_TEAM],
        right_on=[MAP_COL_PLAYER_SHOT, MAP_COL_TEAM_SHOT],
        suffixes=("", "_shootmap"),
    )
    # Overwrite shooter with mapped name, fallback to original name
    df[COL_SHOT_SHOOTER_NAME] = df[MAP_COL_PLAYER_PERF].fillna(original_shooter)

    # Drop helper columns
    for c in [MAP_COL_PLAYER_SHOT, MAP_COL_TEAM_SHOT, MAP_COL_PLAYER_PERF]:
        if c in df.columns:
            df.drop(columns=c, inplace=True, errors='ignore')

    return df


### Cleaning Shot Events & Preventing Data Leakage
Loads shot events, drops disclosing attributes (data leakage) that would reveal goals/saves beforehand, and constructs the target label `Goal`.

In [6]:
print("Loading ISA shot events dataset...")
shots_raw = pd.read_excel("all_shots_check_fixed_raw.xlsx")

# Clean missing rows
shots_df = shots_raw.dropna(how='all').copy()

# Define target variable
shots_df["Goal"] = shots_df["playType"].apply(lambda x: 1 if "GOAL" in str(x).strip().upper() else 0)

# Create is_shooter_home attribute
shots_df['is_shooter_home'] = (shots_df['Team'] == shots_df['homeTeam']).astype(int)

# Define columns to keep (filtering out data leakage / outcome disclosing attributes)
columns_to_keep = [
    "Position", "FieldWidth", "FieldLocation",
    "EventX", "EventY", "GKx", "GKy",
    "homeCurrentScore", "awayCurrentScore",
    "PossNum", "ShotPatternOfPlay", "ShotBodyPart", "Pressure",
    "PlyrsBtwn", "sequenceSeconds", "possessionSeconds",
    "playType", "Team", "Date", "shooter", "goalie", "goalie team", "is_shooter_home", "TimeStamp", "xG", "Goal"
]

shots_df = shots_df[columns_to_keep].copy()

# Convert TimeStamp to standard string format
shots_df["TimeStamp"] = pd.to_datetime(shots_df["TimeStamp"]).dt.strftime('%H:%M:%S')

# Format Date column as datetime
shots_df["Date"] = pd.to_datetime(shots_df["Date"], errors="coerce")

# Apply player name mapping to SofaScore vocabulary
print("Mapping player names to SofaScore names...")
shots_mapped = map_shots_names_to_sofa(shots_df, "ligat_haal_player_match_cleaned.xlsx")

print(f"Shots preprocessed shape: {shots_mapped.shape}")

# Save to CSV
shots_mapped.to_csv("shots_preprocessed.csv", index=False)
print("Saved shots preprocessed data to 'shots_preprocessed.csv'.")


Loading ISA shot events dataset...
Mapping player names to SofaScore names...
Shots preprocessed shape: (11571, 26)
Saved shots preprocessed data to 'shots_preprocessed.csv'.
